# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² clinical dataset using the `mlcroissant` library, referencing all entities by their `@id`. No personal health information is used beyond dataset summary fields.

### Dataset Source
Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and dataset records using `mlcroissant`, referencing the dataset schema by URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata object directly
metadata_obj = dataset.metadata
print(f"Dataset title: {metadata_obj.name}\nDescription: {metadata_obj.description}")

## 2. Data Overview
Explore available record sets, fields, and columns, referencing them by their `@id`.

In [ ]:
# List all available record sets in the dataset by @id with summary
record_sets = dataset.record_sets

print("Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '<unnamed>')}, description: {rs.get('description', '<no description>')}")

# For each record set, print the fields and columns by @id
for rs in record_sets:
    print(f"\nFields/columns of RecordSet '@id': {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            print(f"  Field @id: {field['@id']} name: {field.get('name', '<unnamed>')}, dataType: {field.get('dataType', '')}")
            if 'column' in field:
                columns = field['column'] if isinstance(field['column'], list) else [field['column']]
                for col in columns:
                    print(f"    Column @id: {col['@id']} name: {col.get('name', '<unnamed>')}, dataType: {col.get('dataType', '')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing the record sets and fields by `@id`.

In [ ]:
# Dynamically collect all record_set @ids
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print the columns of the first record set
if record_sets_ids:
    print(f"Columns for record set @id {record_sets_ids[0]}:")
    print(dataframes[record_sets_ids[0]].columns.tolist())
    display(dataframes[record_sets_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on clinical features, normalizing numeric fields, and grouping by attributes. All entities are referenced strictly by `@id`.

In [ ]:
# For demonstration, choose first record set and its numeric field by @id
current_record_set_id = record_sets_ids[0] if record_sets_ids else None

df = dataframes[current_record_set_id]

# Find a numeric field (e.g., 'age_at_diagnosis') by @id
# Assume field @id is 'table1:age_at_diagnosis' (replace as needed from overview printout)
numeric_field_id = None
for rs in dataset.record_sets:
    if rs['@id'] == current_record_set_id and 'field' in rs:
        for field in rs['field'] if isinstance(rs['field'], list) else [rs['field']]:
            if field.get('dataType','').lower() in ('integer', 'float', 'number') and 'age' in field.get('name','').lower():
                numeric_field_id = field['@id']
                break

# If not found, fallback to first numeric column in dataframe
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id:
    threshold = 18
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by a categorical field, e.g., 'infertility' by @id
    group_field_id = None
    for rs in dataset.record_sets:
        if rs['@id'] == current_record_set_id and 'field' in rs:
            for field in rs['field'] if isinstance(rs['field'], list) else [rs['field']]:
                if 'infertility' in field.get('name','').lower():
                    group_field_id = field['@id']
                    break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped statistics of {numeric_field_id}_normalized by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize key numeric data distributions and relationships in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: plot histogram of age at diagnosis
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], bins=5, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {current_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Example: scatter plot age vs another numeric field
other_num_cols = [col for col in df.columns if col != numeric_field_id and pd.api.types.is_numeric_dtype(df[col])]
if len(other_num_cols) >= 1:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=df[numeric_field_id], y=df[other_num_cols[0]])
    plt.title(f"Scatter: {numeric_field_id} vs {other_num_cols[0]}")
    plt.xlabel(numeric_field_id)
    plt.ylabel(other_num_cols[0])
    plt.show()

## 6. Conclusion
We have successfully loaded and explored the FAIR² clinical dataset using `mlcroissant`, referencing all entities by their `@id`. Key record sets and fields were reviewed, filtered, grouped, normalized, and visualized. For detailed clinical analysis, consult original schema documentation and ensure findings are contextualized given the small, non-generalizable cohort.